# G01 v5 — HAM10000/BCN20000 Provenance & Class Audit

Kaggle CPU, Internet ON. Verifies the 25,331-row SHA-locked N01 master, including 2,144 quarantined rows, then keeps only the 23,187 eligible records for the four N01 modeling partitions. R00 exports only the three non-test splits.


## 1. Configuration and download helper

In [1]:
import os,json,hashlib,zipfile,shutil,subprocess,time,re,io
from pathlib import Path
import pandas as pd,numpy as np
SEED=42;CLASSES=['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
INPUT=Path('/kaggle/input' if Path('/kaggle/input').exists() else '/mnt/data')
WORK=Path('/kaggle/working' if Path('/kaggle/working').exists() else '/mnt/data')
def req(ok,msg):
    if not bool(ok):raise RuntimeError(msg)
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''):h.update(b)
    return h.hexdigest()
def writejson(p,x):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)
    tmp=p.with_suffix(p.suffix+'.tmp');tmp.write_text(json.dumps(x,indent=2,allow_nan=False,default=str));tmp.replace(p)
def find_unique(glob_name,root=INPUT,optional=False):
    hits=sorted(p for p in root.rglob(glob_name) if p.is_file())
    if not hits and optional:return None
    req(len(hits)==1,f'Expected exactly one {glob_name}; got {len(hits)}: {hits[:5]}')
    return hits[0]
def download(url,out,min_size=100):
    out=Path(out);out.parent.mkdir(parents=True,exist_ok=True)
    if out.exists() and out.stat().st_size>=min_size:return out
    cmd=['curl','--fail','--location','--retry','6','--retry-delay','5','--connect-timeout','30','--continue-at','-','--output',str(out),url]
    r=subprocess.run(cmd,check=False)
    req(r.returncode==0 and out.is_file() and out.stat().st_size>=min_size,f'Download failed ({r.returncode}): {url}')
    return out
def getzip():
    # Reuse a prior Kaggle notebook output, otherwise download the official ISIC 2019 archive.
    zps=sorted([p for p in INPUT.rglob('ISIC_2019_Training_Input.zip') if p.is_file()])
    zps+=sorted([p for p in (WORK/'g_cache').glob('ISIC_2019_Training_Input.zip') if p.is_file()])
    if zps:archive=zps[0]
    else:
        req(os.getenv('KAGGLE_INTERNET_ENABLED','1')=='1','Enable Kaggle Internet for first download')
        archive=download('https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_Input.zip',WORK/'g_cache/ISIC_2019_Training_Input.zip',min_size=5*10**9)
    with zipfile.ZipFile(archive) as z:
        names={Path(x).stem:x for x in z.namelist() if x.lower().endswith(('.jpg','.jpeg','.png'))}
    req(len(names)==25331,f'Official ISIC 2019 archive should contain 25,331 distinct JPEGs, found {len(names)}')
    return archive,names

NAME='g01-ham-bcn-two-track-audit';OUT=WORK/NAME
for k in ['metadata','manifests','reports','tables']:(OUT/k).mkdir(parents=True,exist_ok=True)
URL={
    'isic19_labels':'https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_GroundTruth.csv',
    'isic19_meta':'https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_Metadata.csv',
    'ham18_lesions':'https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task3_Training_LesionGroupings.csv',
    'bcn249_meta':'https://api.isic-archive.com/collections/249/metadata/'
}
files={k:download(v,OUT/'metadata'/(k+'.csv'),min_size=1000) for k,v in URL.items()}
print({k:(p.stat().st_size,sha(p)[:12]) for k,p in files.items()})

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1261k  100 1261k    0     0  1430k      0 --:--:-- --:--:-- --:--:-- 1429k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1185k  100 1185k    0     0  1260k      0 --:--:-- --:--:-- --:--:-- 1260k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  481k  100  481k    0     0   613k      0 --:--:-- --:--:-- --:--:--  614k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  583k    0  583k    0     0   486k      0 --:--:--  0:00:01 --:--:--  486k

{'isic19_labels': (1291479, 'aa88e9638fe4'), 'isic19_meta': (1214351, 'd93994a8ed20'), 'ham18_lesions': (492996, 'b0352393dc54'), 'bcn249_meta': (3639183, '9ab7c4334df2')}


100 3553k    0 3553k    0     0  2430k      0 --:--:--  0:00:01 --:--:-- 2429k


## 2. Build authoritative eight-class ISIC 2019 labels and source memberships

In [2]:
gt=pd.read_csv(files['isic19_labels'],dtype=str,keep_default_na=False)
meta19=pd.read_csv(files['isic19_meta'],dtype=str,keep_default_na=False)
ham=pd.read_csv(files['ham18_lesions'],dtype=str,keep_default_na=False)
bcn=pd.read_csv(files['bcn249_meta'],dtype=str,keep_default_na=False)

ID_FIELDS=['image','isic_id','image_id','ISIC_id','ImageID','image_name','name','isicid']
ID_PATTERN=r'ISIC_\d+(?:_downsampled)?'
def clean_ids(s):
    s=s.fillna('').astype(str).str.strip().str.strip('"\'').str.replace('\\','/',regex=False)
    s=s.str.split('?',n=1).str[0].str.rsplit('/',n=1).str[-1]
    s=s.str.replace(r'(?i)\.(?:jpg|jpeg|png|dcm|tiff|tif|bmp)$','',regex=True).str.strip().str.upper()
    return s.str.replace(r'_DOWNSAMPLED$','_downsampled',regex=True)
def base_ids(s):return s.astype(str).str.replace(r'_downsampled$','',regex=True)
def canonical(df,source):
    f=df.copy();available=[c for c in ID_FIELDS if c in f.columns]
    req(bool(available),f'{source}: missing image ID column; columns={f.columns.tolist()}')
    v={k:clean_ids(f[k]) for k in available}
    valid={k:v[k].str.fullmatch(ID_PATTERN).fillna(False) for k in available}
    ranked=sorted(available,key=lambda k:(-int(valid[k].sum()),ID_FIELDS.index(k)))
    chosen=ranked[0];ids=v[chosen].copy();filled=0
    for alt in ranked[1:]:
        both=valid[chosen]&valid[alt]
        conflicts=both&(v[chosen]!=v[alt])
        if conflicts.any():
            raise RuntimeError(f'{source}: conflicting image IDs: {f.loc[conflicts,[chosen,alt]].head(8).to_dict("records")}')
        fill=(~ids.str.fullmatch(ID_PATTERN).fillna(False))&valid[alt]
        ids.loc[fill]=v[alt].loc[fill];filled+=int(fill.sum())
    good=ids.str.fullmatch(ID_PATTERN).fillna(False)
    print(f'{source}: n={len(f)} valid={int(good.sum())} invalid={int((~good).sum())} '
          f'downsampled={int(ids.str.endswith("_downsampled").sum())} chosen={chosen} filled={filled}')
    if not good.all():
        bad=f.loc[~good,available].copy();bad.insert(0,'csv_row',bad.index+2)
        p=OUT/'reports'/f'{source}_invalid_ids.csv';bad.to_csv(p,index=False)
        raise RuntimeError(f'{source}: {len(bad)} invalid IDs; inspect {p}; examples={bad.head(8).to_dict("records")}')
    if ids.duplicated().any():
        raise RuntimeError(f'{source}: duplicate full IDs: {ids.loc[ids.duplicated(keep=False)].head(10).tolist()}')
    f['image_id']=ids.to_numpy();return f

ham=canonical(ham,'HAM2018');bcn=canonical(bcn,'BCN249')
gt=canonical(gt,'ISIC2019_GT');meta19=canonical(meta19,'ISIC2019_METADATA')
req(len(gt)==25331,f'ISIC2019 count mismatch: {len(gt)}')
req(set(gt.image_id)==set(meta19.image_id),
    f'GT/metadata IDs mismatch: GT-only={sorted(set(gt.image_id)-set(meta19.image_id))[:8]}, '
    f'metadata-only={sorted(set(meta19.image_id)-set(gt.image_id))[:8]}')
req(all(k in gt.columns for k in CLASSES),'Missing ISIC2019 diagnosis columns')
Y=gt[CLASSES].apply(pd.to_numeric,errors='raise').astype(int).to_numpy()
req(((Y==0)|(Y==1)).all() and (Y.sum(1)==1).all(),'ISIC2019 labels must be one-hot')
gt['label']=np.asarray(CLASSES)[Y.argmax(1)]
req('lesion_id' in meta19.columns,'ISIC2019 metadata missing lesion_id')

# Source IDs are sometimes unsuffixed while the ISIC2019 image/ZIP filename is suffixed.
# Keep the complete image_id as the primary key; base IDs are used ONLY to reconcile source membership.
hkeys=set(base_ids(ham.image_id));bkeys=set(base_ids(bcn.image_id))
shared=hkeys&bkeys
if shared:
    p=OUT/'reports/ham_bcn_shared_base_ids.csv'
    pd.DataFrame({'base_id':sorted(shared)}).to_csv(p,index=False)
    raise RuntimeError(f'{len(shared)} HAM/BCN base IDs overlap; inspect {p}')
for name,src in [('HAM2018',ham),('BCN249',bcn)]:
    keys=base_ids(src.image_id)
    if keys.duplicated().any():
        p=OUT/'reports'/f'{name}_base_collisions.csv'
        src.loc[keys.duplicated(keep=False)].to_csv(p,index=False)
        raise RuntimeError(f'{name}: ambiguous base IDs; inspect {p}')

cohort=gt[['image_id','label']].merge(meta19[['image_id','lesion_id']],on='image_id',validate='one_to_one')
cohort['source_key']=base_ids(cohort.image_id)
cohort['source']='OTHER';cohort['source_match']='none'
exact_h=cohort.image_id.isin(set(ham.image_id));exact_b=cohort.image_id.isin(set(bcn.image_id))
req(not(exact_h&exact_b).any(),'Exact image ID belongs to both source inventories')
cohort.loc[exact_h,['source','source_match']]=['HAM10000','exact']
cohort.loc[exact_b,['source','source_match']]=['BCN20000','exact']
# Do not alias a base ID if two different 2019 images share it.
collided=cohort.source_key.duplicated(keep=False)
needs_alias=cohort.source.eq('OTHER')&cohort.source_key.isin(hkeys|bkeys)
unsafe=needs_alias&collided
if unsafe.any():
    p=OUT/'reports/ambiguous_downsampled_source_aliases.csv'
    cohort.loc[cohort.source_key.isin(set(cohort.loc[unsafe,'source_key'])),
               ['image_id','source_key','source','label']].to_csv(p,index=False)
    raise RuntimeError(f'{int(unsafe.sum())} base-alias matches collide with other ISIC2019 images: {p}')
for source,keys in [('HAM10000',hkeys),('BCN20000',bkeys)]:
    alias=needs_alias&cohort.source_key.isin(keys)
    cohort.loc[alias,['source','source_match']]=[source,'base_alias']
cohort=cohort.loc[cohort.source.ne('OTHER')].copy()
req((cohort.source=='HAM10000').sum()>1000 and (cohort.source=='BCN20000').sum()>1000,
    'Unexpectedly low HAM/BCN intersection; check collection versions')
# HAM lesion metadata may contain only unsuffixed filenames; join by audited source_key.
ham_l=ham[['image_id']+(['lesion_id'] if 'lesion_id' in ham.columns else [])].copy()
ham_l['source_key']=base_ids(ham_l.image_id)
ham_l=ham_l.drop(columns='image_id').rename(columns={'lesion_id':'ham_lesion_id'})
cohort=cohort.merge(ham_l,on='source_key',how='left',validate='many_to_one')
cohort['lesion_id']=cohort.lesion_id.fillna('').astype(str).str.strip()
cohort['lesion_id_known']=cohort.lesion_id.ne('')
# Missing lesion IDs must not be silently collapsed into one pseudo-lesion.
cohort['source_group']=cohort.source+':'+np.where(cohort.lesion_id_known,
    'LESION:'+cohort.lesion_id,'UNKNOWN_IMAGE:'+cohort.image_id)
cohort['label_id']=cohort.label.map({v:i for i,v in enumerate(CLASSES)}).astype(int)
cohort=cohort.sort_values(['source','image_id']).reset_index(drop=True)
cohort.to_csv(OUT/'metadata/source_membership_isic2019.csv',index=False)
print(cohort.groupby(['source','source_match']).size().to_string())
print(cohort.groupby(['source','label']).size().unstack(fill_value=0).to_string())
print('Matched HAM/BCN',len(cohort),'downsampled',int(cohort.image_id.str.endswith('_downsampled').sum()),
      'missing lesion IDs',int((~cohort.lesion_id_known).sum()))


HAM2018: n=10015 valid=10015 invalid=0 downsampled=0 chosen=image filled=0
BCN249: n=18946 valid=18946 invalid=0 downsampled=0 chosen=isic_id filled=0
ISIC2019_GT: n=25331 valid=25331 invalid=0 downsampled=2074 chosen=image filled=0
ISIC2019_METADATA: n=25331 valid=25331 invalid=0 downsampled=2074 chosen=image filled=0
source    source_match
BCN20000  exact           12413
HAM10000  exact           10015
label      AK   BCC   BKL   DF   MEL    NV  SCC  VASC
source                                               
BCN20000  737  2809  1138  124  2857  4206  431   111
HAM10000  130   514  1099  115  1113  6705  197   142
Matched HAM/BCN 22428 downsampled 0 missing lesion IDs 0


## 3. Match original image-level J1 with N01/R00 split provenance (test IDs from N01 master)


In [3]:
# R00 exports only train/meta_train/validation. N01 owns the complete manifest, including test.
orig_path=find_unique('original_80_10_10_locked_ids.csv',optional=True)
orig_json=find_unique('rotation_fixed_train_val_test_split.json',optional=True) if orig_path is None else None
r00_config=find_unique('experiment_protocol.json',optional=True)
orig=None
if orig_path:
    x=pd.read_csv(orig_path,dtype=str,keep_default_na=False)
    req({'image_id','split'}.issubset(x.columns),'J1 original split CSV has unexpected columns')
    orig=x[['image_id','split']].copy()
    orig['split']=orig['split'].str.lower().replace({'val':'validation'})
elif orig_json:
    j=json.loads(orig_json.read_text())
    orig=pd.concat([pd.DataFrame({'image_id':[Path(t['filepath']).stem for t in j[k]],'split':{'train':'train','val':'validation','test':'test'}[k]}) for k in ['train','val','test']],ignore_index=True)

EXPECTED_N01={'train':16490,'meta_train':2267,'validation':2186,'test':2244}
EXPECTED_QUARANTINE=2144
N01_PARTS=list(EXPECTED_N01)
EXPECTED_MASTER={**EXPECTED_N01,'quarantine':EXPECTED_QUARANTINE}

def resolve_n01(expected_sha):
    found=[]
    for p in INPUT.rglob('experiment_master_manifest.csv'):
        if p.is_file() and sha(p)==expected_sha:found.append(p.resolve())
    if not found:
        cache=WORK/'g01_n01_manifest_cache';cache.mkdir(parents=True,exist_ok=True)
        for zp in sorted(INPUT.rglob('*.zip')):
            try:
                with zipfile.ZipFile(zp) as z:
                    members=[i for i in z.infolist() if not i.is_dir() and i.filename.replace('\\','/').endswith('splits/experiment_master_manifest.csv')]
                    for m in members:
                        req(m.file_size<100*2**20,f'Unexpectedly large N01 manifest in {zp}')
                        data=z.read(m)
                        if hashlib.sha256(data).hexdigest()!=expected_sha:continue
                        p=cache/'experiment_master_manifest.csv';p.write_bytes(data);found.append(p.resolve())
            except zipfile.BadZipFile:
                continue
    found=sorted(set(found))
    req(bool(found),'N01 master manifest not found or SHA mismatch. Attach the output of '
        '`lesion-n01-dataset-audit-grouped-splits` containing splits/experiment_master_manifest.csv '
        '(R00 alone does not contain the locked test split).')
    req(all(sha(p)==expected_sha for p in found),'N01 manifest changed after discovery')
    return found[0]

new=None;locked=None;n01_master_path=None
if r00_config:
    root=r00_config.parent.parent
    locked=json.loads(r00_config.read_text())
    req(locked.get('status')=='LOCKED' and locked.get('classes')==CLASSES,'Wrong R00 protocol/classes')
    checks_path=root/'reports/artifact_checksums.json'
    req(checks_path.is_file(),f'Missing R00 checksums: {checks_path}')
    checks=json.loads(checks_path.read_text())
    req(sha(r00_config)==checks.get('configs/experiment_protocol.json'),'R00 protocol checksum mismatch')
    n01_master_path=resolve_n01(locked['n01_manifest_sha256'])
    master=pd.read_csv(n01_master_path,dtype=str,keep_default_na=False)
    req({'image_id','group_id','split'}.issubset(master.columns),
        f'N01 master missing columns; found {master.columns.tolist()}')
    req(master.image_id.is_unique,'N01 master contains duplicate image IDs')
    master['split']=master['split'].str.strip().str.lower().replace({'val':'validation'})
    counts=master['split'].value_counts().to_dict()
    req(set(counts)==set(EXPECTED_MASTER),f'Unexpected N01 split names: {counts}')
    req(len(master)==sum(EXPECTED_MASTER.values()),f'N01 full master count mismatch: {len(master)}')
    for part,n in EXPECTED_MASTER.items():
        req(counts.get(part,0)==n,f'N01 {part}: expected {n}, got {counts.get(part,0)}')
    # Preserve all 25,331 rows in the SHA-verified master, but never put quarantine into model partitions.
    quarantine=master.loc[master.split.eq('quarantine')].copy()
    eligible=master.loc[master.split.isin(N01_PARTS)].copy()
    req(len(eligible)==sum(EXPECTED_N01.values()),'N01 eligible cohort count mismatch')
    req(eligible.group_id.str.strip().ne('').all(),'Eligible N01 partition has missing lesion groups')
    req(set(quarantine.image_id).isdisjoint(set(eligible.image_id)), 'Quarantine overlaps eligible IDs')
    quarantine_ids=set(quarantine.image_id)
    quarantine.to_csv(OUT/'reports/n01_quarantine_audit.csv',index=False)
    parts=[]
    for s in ['train','meta_train','validation']:
        p=root/'splits'/f'{s}.csv'
        req(p.is_file(),f'R00 split missing: {p}')
        req(sha(p)==checks.get(f'splits/{s}.csv'),f'R00 {s} checksum mismatch')
        x=pd.read_csv(p,dtype=str,keep_default_na=False)
        req({'image_id','group_id','split'}.issubset(x.columns),f'R00 {s} missing identity/split columns')
        req(len(x)==EXPECTED_N01[s] and x.image_id.is_unique,f'R00 {s} count/identity mismatch')
        req(x.split.eq(s).all(),f'R00 {s} contains unexpected split labels')
        m=eligible.loc[eligible.split.eq(s)]
        req(set(x.image_id)==set(m.image_id),f'R00 and N01 master disagree on {s} image IDs')
        a=x.set_index('image_id').sort_index();b=m.set_index('image_id').sort_index()
        req(a.group_id.equals(b.group_id),f'R00/N01 {s} lesion group mismatch')
        for k in ['label','class_index','source_sha256']:
            if k in a.columns and k in b.columns:
                req(a[k].equals(b[k]),f'R00/N01 {s} field mismatch: {k}')
        parts.append(x[['image_id','group_id']].assign(n01_split=s))
    test=eligible.loc[eligible.split.eq('test'),['image_id','group_id']].copy()
    parts.append(test.assign(n01_split='test'))
    new=pd.concat(parts,ignore_index=True)
    req(new.image_id.is_unique and len(new)==23187,'N01 joined split identities invalid')
    for s1 in N01_PARTS:
        for s2 in N01_PARTS:
            if N01_PARTS.index(s1)>=N01_PARTS.index(s2):continue
            a=set(new.loc[new.n01_split.eq(s1),'group_id']);b=set(new.loc[new.n01_split.eq(s2),'group_id'])
            req(a.isdisjoint(b),f'N01 group crossed {s1}/{s2}')
    print('VERIFIED N01 eligible partitions:',EXPECTED_N01,'; quarantine excluded:',len(quarantine))

if orig is not None:
    req(orig.image_id.is_unique,'Original split includes duplicate IDs')
    cohort=cohort.merge(orig.rename(columns={'split':'image_split'}),on='image_id',how='left',validate='one_to_one')
else:cohort['image_split']='UNKNOWN'
if new is not None:
    cohort=cohort.merge(new,on='image_id',how='left',validate='one_to_one')
else:cohort['n01_split']='UNKNOWN';cohort['group_id']=''
cohort[['image_split','n01_split']]=cohort[['image_split','n01_split']].fillna('OUTSIDE')
cohort['n01_quarantined']=cohort.image_id.isin(quarantine_ids) if new is not None else False
cohort.loc[cohort.n01_quarantined,'n01_split']='QUARANTINE'
cohort['group_id']=cohort.group_id.fillna('')
req(not (cohort.n01_quarantined & cohort.n01_split.isin(N01_PARTS)).any(),
    'Quarantined images unexpectedly appear in eligible N01 partitions')
cohort['image_test']=cohort.image_split.eq('test')
cohort['n01_validation']=cohort.n01_split.eq('validation')
cohort['n01_locked_test']=cohort.n01_split.eq('test')
cohort['joint_test']=cohort.image_test & cohort.n01_locked_test
cohort['protocol_ready']=bool(orig is not None and new is not None)
cohort.to_csv(OUT/'manifests/all_sources_with_splits.csv',index=False)
for name,condition in [('G02A_original_test',cohort.image_test),('G02B_validation',cohort.n01_validation),('G02B_locked_test',cohort.n01_locked_test),('joint_test',cohort.joint_test)]:
    d=cohort.loc[condition].copy();d.to_csv(OUT/'manifests'/f'{name}.csv',index=False)
    print(name,len(d),d.source.value_counts().to_dict(),d.label.value_counts().to_dict())
print('Found J1:',bool(orig is not None),'Found R00:',bool(new is not None),'N01 master:',str(n01_master_path or ''))


VERIFIED N01 eligible partitions: {'train': 16490, 'meta_train': 2267, 'validation': 2186, 'test': 2244} ; quarantine excluded: 2144
G02A_original_test 0 {} {}
G02B_validation 2109 {'HAM10000': 1060, 'BCN20000': 1049} {'NV': 1121, 'MEL': 308, 'BCC': 289, 'BKL': 218, 'AK': 73, 'SCC': 49, 'VASC': 41, 'DF': 10}
G02B_locked_test 2160 {'BCN20000': 1090, 'HAM10000': 1070} {'NV': 1059, 'MEL': 379, 'BCC': 333, 'BKL': 211, 'AK': 70, 'SCC': 61, 'DF': 25, 'VASC': 22}
joint_test 0 {} {}
Found J1: True Found R00: True N01 master: /kaggle/input/notebooks/rinkyranisaha/lesion-n01/lesion-n01-dataset-audit-grouped-splits/splits/experiment_master_manifest.csv


## 4. Source and overlap report — do not claim these are unseen datasets

In [4]:
table=(cohort.groupby(['source','label','image_split','n01_split'],dropna=False).size().rename('n').reset_index())
table.to_csv(OUT/'tables/class_source_split_counts.csv',index=False)
for s in ['HAM10000','BCN20000']:
    c=cohort[cohort.source.eq(s)]
    print(s,'n=',len(c),'missing lesion IDs=',int(c.lesion_id.eq('').sum()),'original train=',int(c.image_split.eq('train').sum()),'N01 train=',int(c.n01_split.eq('train').sum()))
summary={'stage':'G01','status':'COMPLETED','protocol_ready':bool(orig is not None and new is not None),'classes':CLASSES,
         'scope':'labeled HAM/BCN intersections of ISIC2019 training; NOT independent external datasets',
         'label_policy':'ISIC2019 eight-class ground truth; HAM2018 AKIEC is approximated by AK; no test-label-based class masking in G02',
         'urls':URL,'sha256':{k:sha(v) for k,v in files.items()},'source_counts':cohort.source.value_counts().to_dict(),
         'cohort_counts':{'G02A_original_test':int(cohort.image_test.sum()),'G02B_validation':int(cohort.n01_validation.sum()),'G02B_locked_test':int(cohort.n01_locked_test.sum())},
         'original_split_source':str(orig_path or orig_json or ''),'r00_source':str(r00_config or ''),'n01_master_source':str(n01_master_path or ''),
         'cohort_sha256':sha(OUT/'manifests/all_sources_with_splits.csv'),'test_predictions_generated':False,
         'id_policy':'Preserve ISIC_..._downsampled in image_id; base aliases only for unique source membership',
         'source_match_counts':cohort.source_match.value_counts().to_dict(),
         'downsampled_matched':int(cohort.image_id.str.endswith('_downsampled').sum()),
         'unknown_lesion_ids':int((~cohort.lesion_id_known).sum()),
         'n01_quarantine_count':int(EXPECTED_QUARANTINE),
         'matched_quarantined_count':int(cohort.n01_quarantined.sum())}
writejson(OUT/'reports/g01_status.json',summary)
req(bool(orig is not None and new is not None),'G01 metadata downloaded but J1/R00 split provenance absent. Attach both outputs and rerun before G02.')
print('G01 COMPLETED:',OUT)


HAM10000 n= 10015 missing lesion IDs= 0 original train= 0 N01 train= 6827
BCN20000 n= 12413 missing lesion IDs= 0 original train= 0 N01 train= 9096
G01 COMPLETED: /kaggle/working/g01-ham-bcn-two-track-audit
